# ZerO / ResNet-18: анализ результатов

Notebook читает CSV/JSON и строит **8 панелей**: Figure 5 (3), Figure 6 (3), Figure 8 (2). Обучение, gradient measurements и compression выполняются только в Python-модулях через `run_zero.py`. Здесь нет скачивания данных и изменения checkpoint.

Протокол: [protocol.md](protocol.md); [статья v3](https://arxiv.org/pdf/2110.12661v3); [официальный commit](https://github.com/jiaweizzhao/ZerO-initialization/tree/cf2bf5d7d967958b3bb232755b288d97239896a8). В авторском репозитории нет CIFAR ResNet: неизвестные настройки обозначены реконструкцией.

По умолчанию анализируются фактические локальные smoke-проверки. Они не сравнимы с опубликованной итоговой accuracy.

In [1]:
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
BASE = Path.cwd()
if not (BASE / "zero_experiments").exists():
    BASE = next(p for p in (BASE / "hw2", BASE / "efficient-nn-hw/hw2") if (p / "zero_experiments").exists())
sys.path.insert(0, str(BASE))
from zero_experiments.results import aggregate, read_csv, report
from zero_experiments.plotting import figure5, figure6, figure8, training_curves

# Для полной серии измените эти два пути.
ROOT = BASE / "runs/reproduction-smoke"
GRADIENT_ROOT = BASE / "runs/figure8-smoke"
print(ROOT, GRADIENT_ROOT, sep="\n")

/Users/ilya/Documents/compet/itmo/efficient-nn-hw/hw2/runs/reproduction-smoke
/Users/ilya/Documents/compet/itmo/efficient-nn-hw/hw2/runs/figure8-smoke


## Фактические запуски и статистика

Sample std: ddof=1. 95% CI: Student-t. При n=1 std/CI не определены. Основной checkpoint — последняя эпоха; best observed test — отдельная описательная верхняя оценка, не критерий настройки.

In [2]:
stats, paired, runs = aggregate(ROOT)
print(json.dumps(stats, ensure_ascii=False, indent=2))
print("Парные разности ошибок ZerO−alternative:")
print(json.dumps(paired, ensure_ascii=False, indent=2))
for r in runs:
    print(r["method"], "seed", r["seed"], "epochs", r["epochs"],
          "test error", r["test_error"], "train seconds", r["train_seconds"],
          "process peak RSS MB", r["process_peak_rss_mb"])
if any(r["smoke"] for r in runs):
    display(Markdown("**SMOKE: короткое обучение не является результатом Table 2.**"))

[
  {
    "method": "kaiming",
    "n": 1,
    "mean": 90.625,
    "std": null,
    "ci_low": null,
    "ci_high": null,
    "std_ddof": 1,
    "ci_method": "Student-t 95%",
    "seeds": "0",
    "smoke": true,
    "paper_reference_error": 5.15,
    "paper_reference_std": 0.13,
    "difference_pp": 85.475
  },
  {
    "method": "xavier",
    "n": 1,
    "mean": 89.0625,
    "std": null,
    "ci_low": null,
    "ci_high": null,
    "std_ddof": 1,
    "ci_method": "Student-t 95%",
    "seeds": "0",
    "smoke": true,
    "paper_reference_error": 5.23,
    "paper_reference_std": 0.16,
    "difference_pp": 83.8325
  },
  {
    "method": "zero",
    "n": 1,
    "mean": 90.625,
    "std": null,
    "ci_low": null,
    "ci_high": null,
    "std_ddof": 1,
    "ci_method": "Student-t 95%",
    "seeds": "0",
    "smoke": true,
    "paper_reference_error": 5.13,
    "paper_reference_std": 0.08,
    "difference_pp": 85.495
  }
]
Парные разности ошибок ZerO−alternative:
[
  {
    "comparison": "zer

**SMOKE: короткое обучение не является результатом Table 2.**

## Figure 5 — 3 панели, верхний ряд

Слои `layer2.0.conv1`, `layer3.0.conv1`, `layer4.0.conv1`: первые свёртки второй, третьей и четвёртой групп. Stable rank = Σσ²/σmax². Измерения до обучения и после каждой эпохи; CSV: method, seed, epoch, layer, stable_rank. Ядра сохранены в `rank_weights/epoch-*.pt`.

Развёртка `[Cout,Cin*kH*kW]` выбрана до эксперимента как реконструкция, поскольку авторский conv-analysis код отсутствует. Identity не вычитается. Для нулевого ядра используется документированное значение 0.

In [3]:
fig = figure5(ROOT)
display(fig)
plt.close(fig)

<Figure size 1200x340 with 3 Axes>

## Figure 6 — 3 панели

Слева — stem и 16 residual Conv, без projection shortcuts. Используется **stable rank** по §5; отдельный алгоритм Kernel Rank не опубликован. Mapping `index → layer` сохранён.

В центре — magnitude pruning внутри каждого Conv/Linear независимо от предыдущих точек, из исходного last checkpoint, без fine-tuning. X — **фактический процент ненулевых весов**, убывает слева направо; доля сохранённых mask slots записана отдельно.

Справа — Tucker-2 выбранной `layer4.1.conv2`, `(r_out,r_in)=(r,r)`. Это явно обозначенная реконструкция. Ядро заменено `1×1 → 3×3 → 1×1` с сохранёнными stride/padding/bias. Полноранговая проверка обязательна перед оценкой. MACs/parameters не являются доказательством latency speedup.

In [4]:
fig = figure6(ROOT)
display(fig)
plt.close(fig)
for p in sorted((ROOT / "compression").glob("*/tucker_full_rank_check.json")):
    print(p.parent.name, json.loads(p.read_text()))
for p in sorted((ROOT / "compression").glob("*/tucker.csv")):
    values = read_csv(p)
    if values:
        r = values[0]
        print(p.parent.name, "params:", r["parameters_before"], "→", r["parameters_after"],
              "MACs/image:", r["macs_before"], "→", r["macs_after"])

<Figure size 1200x340 with 3 Axes>

resnet18-kaiming-seed0 {'max_abs_error': 2.1696090698242188e-05, 'atol': 3.110903739929199e-05, 'rtol': 0.0002, 'output_scale': 6.221807479858398, 'relative_frobenius_error': 1.8106020434061065e-06, 'r_out': 512, 'r_in': 512}
resnet18-xavier-seed0 {'max_abs_error': 1.239776611328125e-05, 'atol': 2.2742204666137696e-05, 'rtol': 0.0002, 'output_scale': 4.548440933227539, 'relative_frobenius_error': 1.8326375084143365e-06, 'r_out': 512, 'r_in': 512}
resnet18-zero-seed0 {'max_abs_error': 0.00060272216796875, 'atol': 0.0011287064361572266, 'rtol': 0.0002, 'output_scale': 225.7412872314453, 'relative_frobenius_error': 1.7982167719310382e-06, 'r_out': 512, 'r_in': 512}
resnet18-kaiming-seed0 params: 11173962 → 8838746 MACs/image: 555422720 → 518059264
resnet18-xavier-seed0 params: 11173962 → 8838746 MACs/image: 555422720 → 518059264
resnet18-zero-seed0 params: 11173962 → 8838746 MACs/image: 555422720 → 518059264


## Figure 8 — 2 панели

Шесть коротких запусков: три метода × warmup/no-warmup. Initial LR=.01/.1. Измеряется `conv1.weight.grad` после backward до step, без clipping, FP32. Норма L2/Frobenius и точное линейное warmup-расписание — предположения. Полный протокол требует 400 итераций каждой группы; локальная проверка содержит меньше точек.

In [5]:
fig = figure8(ROOT, GRADIENT_ROOT)
display(fig)
plt.close(fig)
from collections import defaultdict
counts, hashes = defaultdict(int), defaultdict(set)
for r in read_csv(GRADIENT_ROOT / "gradients.csv"):
    counts[(r["method"], r["mode"], r["seed"])] += 1
    hashes[(r["seed"], r["step"])].add(r["batch_sha256"])
print("Фактически измерено:", dict(counts))
assert all(len(v) == 1 for v in hashes.values()), "Batches/augmentation differ"
print("Все сохранённые batch hashes совпадают:", bool(hashes))

<Figure size 1000x340 with 2 Axes>

Фактически измерено: {('zero', 'warmup', '0'): 8, ('kaiming', 'warmup', '0'): 8, ('xavier', 'warmup', '0'): 8, ('zero', 'no_warmup', '0'): 8, ('kaiming', 'no_warmup', '0'): 8, ('xavier', 'no_warmup', '0'): 8}
Все сохранённые batch hashes совпадают: True


## Кривые обучения и стоимость вычислений

In [6]:
fig = training_curves(ROOT)
display(fig)
plt.close(fig)

<Figure size 1200x340 with 3 Axes>

## Интерпретация

Небольшое отличие средних не доказывает превосходство метода. Stable rank не равен numerical rank или Tucker rank; низкий stable rank не доказывает механизм GLRL. Возможные причины расхождений — неизвестные архитектура/preprocessing/scheduler авторов, версии библиотек, режим детерминизма и короткое обучение.

Figure 4 с изменением глубины, ResNet-50, MNIST и Transformer исключены. Недостающие результаты не дорисовываются.

In [7]:
path = report(ROOT, GRADIENT_ROOT)
display(Markdown(path.read_text()))
print("PNG/PDF:", ROOT / "plots")
print("CSV/JSON и отчёт:", ROOT)

# Фактический отчёт ResNet-18 / CIFAR-10

Протокол и источники: [protocol.md](../../protocol.md). Все числа ниже взяты из сохранённых измерений.
Реконструкция содержит неизвестные авторские настройки; совпадение accuracy не подтверждает точное воспроизведение.

| Эксперимент | Фактическое покрытие |
|---|---|
| Table 2 | 0/30 полных запусков; дополнительно 3 smoke |
| Table 3 | 0 запусков альтернатив |
| Figure 5 | 3 траекторий, включая smoke |
| Figure 6 | 3 обработанных checkpoint, статус в metadata.json |
| Figure 8 | 6 групп; 48 шагов; 0 групп с 400 шагами |

Std — sample ddof=1; CI — 95% Student-t; при n=1 std и CI не определены. Smoke не является результатом Table 2.

| Метод | n | Mean test error | Sample std | 95% CI | Ориентир статьи | Smoke |
|---|---:|---:|---:|---|---:|---|
| kaiming | 1 | 90.6250 | None | [None, None] | 5.15 | True |
| xavier | 1 | 89.0625 | None | [None, None] | 5.23 | True |
| zero | 1 | 90.6250 | None | [None, None] | 5.13 | True |

## Отдельные запуски

| Метод | Seed | Эпохи | Last error | Best observed test accuracy | Train seconds | Peak RSS MB |
|---|---:|---:|---:|---:|---:|---:|
| kaiming | 0 | 2 | 90.6250 | 9.3750 | 4.21 | 966.5 |
| xavier | 0 | 2 | 89.0625 | 10.9375 | 3.79 | 1037.1 |
| zero | 0 | 2 | 90.6250 | 9.3750 | 3.88 | 786.1 |

Best observed test — описательная верхняя оценка, не основной checkpoint и не tuning criterion.
Ранги, test evaluation и checkpoint IO исключены из train_seconds; rank_seconds и test_seconds сохранены отдельно.
RSS — high-water mark всего процесса, CUDA memory — max_memory_allocated; это разные измерения.

## Figure 8

| Метод | Режим | Seed | Шагов | Первый LR |
|---|---|---:|---:|---:|
| kaiming | no_warmup | 0 | 8 | 0.1 |
| kaiming | warmup | 0 | 8 | 0.01 |
| xavier | no_warmup | 0 | 8 | 0.1 |
| xavier | warmup | 0 | 8 | 0.01 |
| zero | no_warmup | 0 | 8 | 0.1 |
| zero | warmup | 0 | 8 | 0.01 |

## Графики

![figure8-resnet18.png](plots/figure8-resnet18.png)
![figure5-resnet18.png](plots/figure5-resnet18.png)
![figure6-resnet18.png](plots/figure6-resnet18.png)
![training.png](plots/training.png)

## Интерпретация и оставшиеся эксперименты

Небольшое отличие средних не означает превосходства. Парные разности/CI и отношение std — в paired.csv; при разных наборах seed или smoke вывод о воспроизводимости ограничен.
Stable rank не доказывает механизм GLRL. Kernel-rank панель использует выбранный stable rank; точный алгоритм Figure 6 не опубликован.
Tucker сообщает параметры и MACs, не обещает ускорения. Полноранговые проверки записаны отдельно. Pruning не содержит fine-tuning.
Основная серия: недостающих полных запусков 30 из 30. Полная Figure 8 требует всех трёх методов в обоих режимах, 400 шагов каждой группы. Альтернативы и compression подтверждаются только записями соответствующих каталогов.
Figure 4/depth, ResNet-50, ImageNet, MNIST и Transformer исключены по постановке.
Возможные причины расхождений: неизвестные architecture/preprocessing/scheduler/weight-decay правила авторов, аппаратный nondeterminism, версия библиотек и короткое обучение.


PNG/PDF: /Users/ilya/Documents/compet/itmo/efficient-nn-hw/hw2/runs/reproduction-smoke/plots
CSV/JSON и отчёт: /Users/ilya/Documents/compet/itmo/efficient-nn-hw/hw2/runs/reproduction-smoke
